<a href="https://colab.research.google.com/github/seonilj/eeg-mne-pipeline/blob/main/notebooks/03_eeg_epoching_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Motor Imagery EEG Epoching

This notebook focuses on the 3rd stage of the EEG pipeline: converting continuous EEG recordings into task-specific epochs, and exploring their basic data structure before machine learning.

Continuous EEG recordings are segmented into short time windows (epochs) according to experimental events. These segmented trials provide the standard input format for subsequent feature extraction and classification algorithms.

In [1]:
# ==============================================================================
# Step 1 & 2: Environment Setup & Loading Filtered EEG (Reused from Notebook 1)
# ==============================================================================

!pip install mne matplotlib

import mne
import matplotlib.pyplot as plt
import numpy as np

print("Loading EEG dataset...")

Loading EEG dataset...


In [2]:
# ==============================================================================
# Reload the same EEG dataset
# ==============================================================================

subject = 1
runs = [4]

raw_fnames = mne.datasets.eegbci.load_data(subject, runs)

raw = mne.io.read_raw_edf(
    raw_fnames[0],
    preload=True
)

# Channel name standardization
mapping = {ch: ch.rstrip('.') for ch in raw.ch_names}
raw.rename_channels(mapping)

# Apply montage
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, match_case=False)

# Repeat preprocessing from Notebook 02
raw.notch_filter(freqs=60, verbose=False)
raw.filter(l_freq=1.0,
           h_freq=40.0,
           verbose=False)

print("Filtered EEG loaded successfully.")

Using default location ~/mne_data for EEGBCI...
Creating /root/mne_data


Do you want to set the path:
    /root/mne_data
as the default EEGBCI dataset path in the mne-python config [y]/n? y
Attempting to create new mne-python configuration file:
/root/.mne/mne-python.json
Could not read the /root/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
Download complete in 24s (2.5 MB)
Extracting EDF parameters from /root/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Filtered EEG loaded successfully.


In [3]:
# ==============================================================================
# Step 8: Segment Continuous EEG into Individual Trials
# ==============================================================================

events, event_id = mne.events_from_annotations(raw)

print("Detected Event Types:")
print(event_id)


# Select only Motor Imagery events
motor_event_id = {
    "left_hand": event_id["T1"],
    "right_hand": event_id["T2"]
}

# Define epoch window
tmin = -0.5
tmax = 4.0

epochs = mne.Epochs(
    raw,
    events,
    event_id=motor_event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=(None,0),
    preload=True,
    verbose=False
)
print(epochs)

print(f"Number of trials: {len(epochs)}")
print(f"Channels: {len(epochs.ch_names)}")
print(f"Time points per trial: {epochs.get_data().shape[-1]}")

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Detected Event Types:
{np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
<Epochs | 15 events (all good), -0.5 – 4 s (baseline -0.5 – 0 s), ~5.4 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7>
Number of trials: 15
Channels: 64
Time points per trial: 721
